# Model Benchmark — vLLM Throughput on Kaggle T4 x2

Benchmarks candidate replacement models for the NBME inference ensemble.
Tests vLLM compatibility, VRAM usage, and tokens/sec on T4 hardware.

**Run with: T4 x2, internet ON** (downloads models from HuggingFace).

| Candidate | Params | Float16 VRAM | TP needed |
|-----------|--------|-------------|----------|
| `Qwen/Qwen3-4B` | 4B | ~8 GB | TP=1 (1 T4) |
| `Qwen/Qwen3-8B` | 8B | ~16 GB | TP=2 (2 T4) |
| `Qwen/Qwen3-4B-AWQ` | 4B | ~3 GB | TP=1 (1 T4) |
| `Qwen/Qwen3-8B-AWQ` | 8B | ~5 GB | TP=1 (1 T4) |
| `meta-llama/Llama-3.1-8B-Instruct` | 8B | ~16 GB | TP=2 (2 T4) |

**Benchmark workload** mirrors actual NBME inference:
- Input: ~600 tokens (system prompt + patient note + feature)
- Output: ~40 tokens (short JSON `{"spans": [...]}`)  
- Greedy decoding (`temperature=0`)
- Batch: 100 prompts (representative sample)

In [1]:
import subprocess, sys
from importlib import metadata

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args, "-q"])

def installed_version(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

# torch/torchvision/torchaudio: keep Kaggle defaults — never downgrade.
# vLLM 0.17.1: pin transformers >=4.56.0,<5 and protobuf < 6 for compatibility.
WANT = {
    "torch":        "2.10.0+cu128",
    "torchvision":  "0.25.0+cu128",
    "torchaudio":   "2.10.0+cu128",
    "transformers": "4.56.0",
    "vllm":         "0.17.1",
}

needs_install = []
for pkg, want in WANT.items():
    have = installed_version(pkg)
    if have == want:
        print(f"  ✓ {pkg} {have} — skip")
    else:
        print(f"  ✗ {pkg} have={have} want={want} — will install")
        needs_install.append(pkg)

if not needs_install:
    print("\nAll packages correct — no install needed.")
else:
    print(f"\nInstalling: {needs_install}")

    if "transformers" in needs_install:
        pip("install", "transformers==4.56.0", "protobuf<6")

    if "vllm" in needs_install:
        pip(
            "install",
            "vllm==0.17.1",
            "protobuf<6",
            "--extra-index-url",
            "https://download.pytorch.org/whl/cu128",
        )
        # Re-pin after vLLM in case the resolver pulled protobuf 6.x.
        pip("install", "transformers==4.56.0", "protobuf<6")

    torch_pkgs = {"torch", "torchvision", "torchaudio"}
    if torch_pkgs & set(needs_install):
        pip("uninstall", "-y", "torch", "torchvision", "torchaudio")
        pip(
            "install",
            "torch==2.10.0",
            "torchvision==0.25.0",
            "torchaudio==2.10.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu128",
        )

    # Final safety pin.
    pip("install", "protobuf<6")

    print("\n✓ Install done.")
    print("⚠ Restarting kernel to flush stale modules ...")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

# Reach here only if all correct — no restart needed
import torch, transformers, vllm
print(f"\ntorch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"vllm:         {vllm.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f}GB  CC={p.major}.{p.minor}")


  ✓ torch 2.10.0+cu128 — skip
  ✓ torchvision 0.25.0+cu128 — skip
  ✓ torchaudio 2.10.0+cu128 — skip
  ✓ transformers 4.56.0 — skip
  ✓ vllm 0.17.1 — skip

All packages correct — no install needed.

torch:        2.10.0+cu128
transformers: 4.56.0
vllm:         0.17.1
CUDA avail:   True
  GPU 0: Tesla T4  14.6GB  CC=7.5
  GPU 1: Tesla T4  14.6GB  CC=7.5


In [2]:
pip("install", "protobuf<6")

In [3]:
import os
import sys
import platform
import subprocess
import torch

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT).decode().strip()
    except Exception as e:
        return f"Error: {e}"

print(f"torch version: {torch.__version__}")
print(f"torch built with CUDA: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(run_cmd("nvidia-smi"))
print(run_cmd("nvcc --version"))
print(torch.__config__.show())

torch version: 2.10.0+cu128
torch built with CUDA: 12.8
CUDA available: True
Fri May  1 16:14:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                    

In [ ]:
import gc, json, os, time
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch

# ── Benchmark config ──────────────────────────────────────────────────────────
N_WARMUP     = 5     # warmup requests (excluded from timing)
N_BENCH      = 500   # benchmark requests
MAX_IN_TOKS  = 1024   # max input tokens (mirrors real prompt length)
MAX_OUT_TOKS = 1024    # max new tokens (clinical spans are short JSON)
GPU_MEM_UTIL = 0.90  # vLLM gpu_memory_utilization

HF_TOKEN = ""
if HF_TOKEN:
    # Write token so HF hub picks it up without login()
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF_TOKEN set — gated repos accessible")
else:
    print("HF_TOKEN not set — gated repos will fail (Llama models skipped)")

# ── Candidate models ──────────────────────────────────────────────────────────
# All ungated except Llama (needs HF_TOKEN + access grant).
# Llama entries kept but will LOAD_ERROR gracefully if token absent.
CANDIDATES = [
    # Small ungated baselines
    {"label": "Qwen3-0.6B (fp16, TP=1)",      "model_id": "Qwen/Qwen3-0.6B",                     "tp": 1, "dtype": "half"},
    {"label": "Qwen3-1.7B (fp16, TP=1)",       "model_id": "Qwen/Qwen3-1.7B",                     "tp": 1, "dtype": "half"},
    {"label": "LiquidAI/LFM2.5-1.2B-Instruct (fp16, TP=1)",       "model_id": "LiquidAI/LFM2.5-1.2B-Instruct",                     "tp": 1, "dtype": "half"},
    {"label": "meta-llama/Llama-3.2-1B-Instruct (fp16, TP=1)",       "model_id": "meta-llama/Llama-3.2-1B-Instruct",                     "tp": 1, "dtype": "half"},
    
    
    # 4B tier — main candidates
    {"label": "Qwen3-4B (fp16, TP=1)",         "model_id": "Qwen/Qwen3-4B",                       "tp": 1, "dtype": "half"},
    {"label": "microsoft/Phi-4-mini-instruct (fp16, TP=1)",         "model_id": "microsoft/Phi-4-mini-instruct",                       "tp": 1, "dtype": "half"},
    
    
    # {"label": "Qwen3-4B-AWQ (int4, TP=1)",     "model_id": "Qwen/Qwen3-4B-AWQ",                   "tp": 1, "dtype": "half"},
    # 7-8B tier — fits TP=1 only with AWQ; fp16 needs TP=2
    # {"label": "Qwen2.5-7B (fp16, TP=2)",       "model_id": "Qwen/Qwen2.5-7B-Instruct",            "tp": 2, "dtype": "half"},
    # {"label": "Qwen3-8B-AWQ (int4, TP=1)",     "model_id": "Qwen/Qwen3-8B-AWQ",                   "tp": 1, "dtype": "half"},
    {"label": "Qwen3-8B (fp16, TP=2)",         "model_id": "Qwen/Qwen3-8B",                       "tp": 2, "dtype": "half"},
    # Gated — only runs if HF_TOKEN set and access granted
    {"label": "Llama-3.1-8B (fp16, TP=2)",    "model_id": "meta-llama/Llama-3.1-8B-Instruct",    "tp": 2, "dtype": "half"},
]

print(f"Candidates: {len(CANDIDATES)}")
print(f"Benchmark:  {N_WARMUP} warmup + {N_BENCH} timed requests")
print(f"Max input:  {MAX_IN_TOKS} tok | Max output: {MAX_OUT_TOKS} tok")

HF_TOKEN set — gated repos accessible
Candidates: 8
Benchmark:  5 warmup + 500 timed requests
Max input:  1024 tok | Max output: 1024 tok


## Section 1 — Synthetic Benchmark Prompts

Generate prompts that match the real NBME inference format:
system prompt + patient note (~400 chars) + clinical feature + `/no_think` suffix.
No real patient data needed — synthetic notes of similar length.

In [5]:
SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation.\n"
    '{"spans": ["exact text 1", "exact text 2"]}'
)

# Synthetic patient notes and features — same length distribution as real NBME data
SAMPLE_NOTES = [
    "Patient is a 45-year-old male presenting with substernal chest pressure radiating to the left arm, onset 2 hours ago. Associated with diaphoresis and mild shortness of breath. Denies nausea or vomiting. History of hypertension, currently on lisinopril. Smoker, 1 pack/day for 20 years. Family history positive for MI in father at age 52. Vitals: BP 158/94, HR 102, RR 18, O2 sat 97% on room air. EKG shows ST elevation in leads II, III, aVF.",
    "24-year-old female presents with severe right lower quadrant pain for 12 hours, nausea, and one episode of vomiting. Pain began periumbilically and migrated to RLQ. Temperature 38.2C, WBC 14.5. Rebound tenderness positive at McBurney's point. LMP was 3 weeks ago. Urine pregnancy test negative. Appetite decreased since yesterday. Pain is 8/10, constant, worsened with movement.",
    "67-year-old female with progressive shortness of breath over 3 weeks, bilateral lower extremity edema, and orthopnea. Requires 3 pillows to sleep. Denies chest pain. History of diabetes mellitus type 2, managed with metformin. Exam shows JVD, crackles at lung bases bilaterally, S3 gallop. Weight gain of 8 lbs in 2 weeks. BNP elevated at 890. Ejection fraction 30% on echo.",
    "19-year-old male with 3-day history of sore throat, fever to 39C, and difficulty swallowing. Tonsillar exudates present bilaterally with tender anterior cervical lymphadenopathy. Rapid strep test positive. Denies cough. No penicillin allergy. Reports eating less than usual due to throat pain. Contact with sick classmate last week.",
    "52-year-old female with 6-month history of fatigue, cold intolerance, constipation, and weight gain of 15 lbs despite no dietary changes. Hair thinning noted. Skin feels dry. TSH 12.4, free T4 low. Reflexes delayed on exam. Denies chest pain or dyspnea. Last menstrual period 2 years ago. Reports feeling depressed and forgetful recently.",
]

SAMPLE_FEATURES = [
    "chest pain or discomfort",
    "nausea or vomiting",
    "shortness of breath",
    "decreased appetite",
    "fever",
    "fatigue or weakness",
    "weight changes",
    "hypertension history",
    "family history of cardiac disease",
    "abdominal pain location",
]

def make_prompt_text(note: str, feature: str) -> str:
    """Build raw text prompt (no chat template — used for vLLM with tokenizer)."""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f'Note: "{note.strip()}"\n'
        f"Feature: {feature}\n\n/no_think<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# Generate N_WARMUP + N_BENCH prompts by cycling through samples
import itertools
all_combos = list(itertools.product(SAMPLE_NOTES, SAMPLE_FEATURES))
total_needed = N_WARMUP + N_BENCH
# Cycle if needed
repeated = (all_combos * (total_needed // len(all_combos) + 1))[:total_needed]
ALL_PROMPTS = [make_prompt_text(note, feat) for note, feat in repeated]

print(f"Generated {len(ALL_PROMPTS)} prompts")
print(f"Example prompt length: {len(ALL_PROMPTS[0])} chars")
print(f"\n--- Example prompt (first 400 chars) ---")
print(ALL_PROMPTS[0][:400])

Generated 505 prompts
Example prompt length: 969 chars

--- Example prompt (first 400 chars) ---
<|im_start|>system
You are a clinical NLP specialist. Given a patient note and a clinical feature, extract the EXACT verbatim text spans from the note that express that feature. Rules:
  1. Copy text character-for-character — do NOT paraphrase.
  2. If the feature is absent from the note, return an empty list.
  3. Output ONLY valid JSON — no markdown, no explanation.
{"spans": ["exact text 1", "e


## Section 2 — Benchmark Runner

For each candidate:
1. Launch vLLM `LLM` engine
2. Run N_WARMUP requests (excluded from timing)
3. Run N_BENCH requests, measure wall time
4. Record: tok/s, VRAM used, time to complete, errors
5. Destroy engine, clear VRAM

Also records whether vLLM loads without error (compatibility check).

In [6]:
import contextlib, shutil, subprocess
from vllm import LLM, SamplingParams
from vllm.config import AttentionConfig
from vllm.v1.attention.backends.registry import AttentionBackendEnum

try:
    from vllm.distributed.parallel_state import destroy_model_parallel
except ImportError:
    def destroy_model_parallel(): pass

MODEL_CACHE_DIR = Path("/kaggle/working/model_cache")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def disk_free_gb(path="/kaggle/working") -> float:
    s = shutil.disk_usage(path)
    return s.free / 1024**3


def disk_used_by_cache_gb() -> float:
    if not MODEL_CACHE_DIR.exists():
        return 0.0
    return sum(f.stat().st_size for f in MODEL_CACHE_DIR.rglob("*") if f.is_file()) / 1024**3


def hf_cache_dir(model_id: str) -> Path:
    """Return the HuggingFace snapshot cache dir for a model_id."""
    org_model = model_id.replace("/", "--")
    return MODEL_CACHE_DIR / f"models--{org_model}"


def get_vram_free_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return sum(torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())) / 1024**3


def get_vram_used_gb() -> float:
    """Query total GPU memory USED via nvidia-smi — visible across subprocesses (vLLM workers)."""
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True,
        )
        return round(sum(int(x.strip()) for x in out.strip().split("\n")) / 1024, 2)
    except Exception:
        return 0.0


def destroy_engine(llm):
    """Destroy vLLM engine and free all GPU memory."""
    destroy_model_parallel()
    with contextlib.suppress(Exception):
        torch.distributed.destroy_process_group()
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def delete_model_cache(model_id: str):
    """Delete downloaded model files from disk to free space for next model."""
    cache_dir = hf_cache_dir(model_id)
    if cache_dir.exists():
        size_gb = sum(f.stat().st_size for f in cache_dir.rglob("*") if f.is_file()) / 1024**3
        shutil.rmtree(cache_dir)
        print(f"  Deleted cache: {cache_dir.name}  ({size_gb:.1f} GB freed)")
    else:
        print(f"  Cache dir not found (already clean): {cache_dir}")


def run_benchmark(candidate: dict, prompts: list, n_warmup: int, n_bench: int) -> dict:
    label    = candidate["label"]
    model_id = candidate["model_id"]
    tp       = candidate["tp"]
    dtype    = candidate["dtype"]

    result = {
        "label":           label,
        "model_id":        model_id,
        "tp":              tp,
        "dtype":           dtype,
        "status":          "PENDING",
        "error":           None,
        "vram_model_gb":   None,
        "vram_peak_gb":    None,
        "output_toks":     None,
        "wall_time_s":     None,
        "output_toks_ps":  None,
        "input_toks_ps":   None,
        "eta_14k_hours":   None,
        "eta_worst_hours": None,
    }

    sp             = SamplingParams(temperature=0.0, max_tokens=MAX_OUT_TOKS)
    warmup_prompts = prompts[:n_warmup]
    bench_prompts  = prompts[n_warmup:n_warmup + n_bench]
    attn_cfg       = AttentionConfig(backend=AttentionBackendEnum.TRITON_ATTN)

    vram_baseline = get_vram_used_gb()
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  model_id: {model_id}  tp={tp}  dtype={dtype}")
    print(f"  Disk free: {disk_free_gb():.1f} GB  |  Cache used: {disk_used_by_cache_gb():.1f} GB")
    print(f"  VRAM used (baseline): {vram_baseline:.2f} GB")
    print(f"{'='*60}")

    # ── Load engine ───────────────────────────────────────────────
    try:
        print(f"  Loading engine ...")
        t_load_start = time.time()
        llm = LLM(
            model                  = model_id,
            dtype                  = dtype,
            tensor_parallel_size   = tp,
            gpu_memory_utilization = GPU_MEM_UTIL,
            max_model_len          = MAX_IN_TOKS + MAX_OUT_TOKS,
            enforce_eager          = True,
            trust_remote_code      = False,
            seed                   = 42,
            download_dir           = str(MODEL_CACHE_DIR),
            attention_config       = attn_cfg,
        )
        t_load_end = time.time()
        vram_after_load = get_vram_used_gb()
        result["vram_model_gb"] = round(vram_after_load - vram_baseline, 2)
        print(f"  Engine loaded in {t_load_end - t_load_start:.1f}s")
        print(f"  VRAM used after load: {vram_after_load:.2f} GB  (model used: {result['vram_model_gb']:.2f} GB)")
    except Exception as e:
        result["status"] = "LOAD_ERROR"
        result["error"]  = str(e)[:300]
        print(f"  LOAD FAILED: {e}")
        delete_model_cache(model_id)
        return result

    # ── Warmup ────────────────────────────────────────────────────
    try:
        print(f"  Warmup ({n_warmup} requests) ...")
        _ = llm.generate(warmup_prompts, sp, use_tqdm=False)
        print(f"  Warmup done.")
    except Exception as e:
        result["status"] = "WARMUP_ERROR"
        result["error"]  = str(e)[:300]
        print(f"  WARMUP FAILED: {e}")
        destroy_engine(llm)
        delete_model_cache(model_id)
        return result

    # ── Benchmark ─────────────────────────────────────────────────
    try:
        print(f"  Benchmarking ({n_bench} requests) ...")
        t0 = time.perf_counter()
        outputs = llm.generate(bench_prompts, sp, use_tqdm=True)
        t1 = time.perf_counter()

        wall_time = t1 - t0
        result["vram_peak_gb"] = round(get_vram_used_gb(), 2)

        total_out_toks = sum(
            sum(len(o.token_ids) for o in req.outputs) for req in outputs
        )
        avg_in_chars    = np.mean([len(p) for p in bench_prompts])
        est_in_toks     = int(avg_in_chars / 4) * n_bench

        result["output_toks"]    = total_out_toks
        result["wall_time_s"]    = round(wall_time, 2)
        result["output_toks_ps"] = round(total_out_toks / wall_time, 1)
        result["input_toks_ps"]  = round(est_in_toks / wall_time, 1)

        FULL_ROWS   = 14_300
        OUT_PER_ROW = 40
        total_out   = FULL_ROWS * OUT_PER_ROW * 3
        eta_best    = total_out / result["output_toks_ps"] / 3600
        eta_worst   = (total_out / (result["output_toks_ps"] * 0.8)) / 3600

        result["eta_14k_hours"]   = round(eta_best, 2)
        result["eta_worst_hours"] = round(eta_worst, 2)
        result["status"]          = "OK"

        print(f"  Wall time:     {wall_time:.1f}s for {n_bench} requests")
        print(f"  Output tokens: {total_out_toks}  ({result['output_toks_ps']} tok/s)")
        print(f"  Peak VRAM:     {result['vram_peak_gb']:.2f} GB")
        print(f"  ETA 14k×3:     {eta_best:.1f}h best / {eta_worst:.1f}h worst")

        print(f"\n  Sample outputs (first 3):")
        for i, req in enumerate(outputs[:3]):
            print(f"    [{i}] {req.outputs[0].text.strip()[:120]}")

    except Exception as e:
        result["status"] = "INFERENCE_ERROR"
        result["error"]  = str(e)[:300]
        print(f"  INFERENCE FAILED: {e}")

    # ── GPU cleanup ───────────────────────────────────────────────
    destroy_engine(llm)
    print(f"  VRAM free after engine destroy: {get_vram_free_gb():.1f} GB")

    # ── Disk cleanup ──────────────────────────────────────────────
    delete_model_cache(model_id)
    print(f"  Disk free after cache delete:   {disk_free_gb():.1f} GB")

    return result


print("✓ Benchmark runner defined")

2026-05-01 16:14:33.802610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777652074.005121     186 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777652074.066306     186 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777652074.538940     186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777652074.538978     186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777652074.538981     186 computation_placer.cc:177] computation placer alr

✓ Benchmark runner defined


## Section 3 — Run All Benchmarks

Runs sequentially. Each model loaded, benchmarked, then destroyed before next loads.
Results saved to `benchmark_results.json` after each model (resume-safe).

In [7]:
RESULTS_PATH = Path("/kaggle/working/benchmark_results.json")

# Load existing results if resuming
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    done_labels = {r["label"] for r in all_results}
    print(f"Resuming — already done: {done_labels}")
else:
    all_results = []
    done_labels = set()

for candidate in CANDIDATES:
    if candidate["label"] in done_labels:
        print(f"SKIP (already done): {candidate['label']}")
        continue

    result = run_benchmark(candidate, ALL_PROMPTS, N_WARMUP, N_BENCH)
    all_results.append(result)

    # Save after each model (resume-safe)
    with open(RESULTS_PATH, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  ✓ Saved results → {RESULTS_PATH}")

print("\n✓ All benchmarks complete")


  Qwen3-0.6B (fp16, TP=1)
  model_id: Qwen/Qwen3-0.6B  tp=1  dtype=half
  Disk free: 19.5 GB  |  Cache used: 0.0 GB
  VRAM used (baseline): 0.01 GB
  Loading engine ...
INFO 05-01 16:14:59 [utils.py:238] non-default args: {'download_dir': '/kaggle/working/model_cache', 'dtype': 'half', 'seed': 42, 'max_model_len': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=True, use_trtllm_attention=None, disable_flashinfer_prefill=False, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False)}


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

INFO 05-01 16:15:21 [model.py:531] Resolved architecture: Qwen3ForCausalLM
WARNING 05-01 16:15:21 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 05-01 16:15:21 [model.py:1554] Using max model len 2048
INFO 05-01 16:15:22 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-01 16:15:22 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 05-01 16:15:22 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-01 16:15:22 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-01 16:15:22 [vllm.py:957] Cudagraph is disabled under eager mode


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:09<00:27,  9.19s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:19<00:19,  9.56s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:34<00:12, 12.18s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:38<00:00,  9.15s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:38<00:00,  9.70s/it]
(Worker pid=5031) (Worker_TP0 pid=5031) 


(Worker pid=5031) (Worker_TP0 pid=5031) INFO 05-01 16:33:36 [default_loader.py:293] Loading weights took 38.84 seconds
(Worker pid=5031) (Worker_TP0 pid=5031) INFO 05-01 16:33:37 [gpu_model_runner.py:4364] Model loading took 7.51 GiB memory and 113.478319 seconds
(Worker pid=5031) (Worker_TP0 pid=5031) INFO 05-01 16:33:47 [gpu_worker.py:424] Available KV cache memory: 5.04 GiB
(EngineCore_DP0 pid=5006) INFO 05-01 16:33:47 [kv_cache_utils.py:1314] GPU KV cache size: 82,592 tokens
(EngineCore_DP0 pid=5006) INFO 05-01 16:33:47 [kv_cache_utils.py:1319] Maximum concurrency for 2,048 tokens per request: 40.33x
(EngineCore_DP0 pid=5006) INFO 05-01 16:33:47 [core.py:282] init engine (profile, create kv cache, warmup model) took 9.51 seconds
(EngineCore_DP0 pid=5006) INFO 05-01 16:33:51 [vllm.py:747] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=5006) WARNING 05-01 16:33:51 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=

Rendering prompts:   0%|          | 0/500 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/500 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Wall time:     17.2s for 500 requests
  Output tokens: 8620  (500.3 tok/s)
  Peak VRAM:     27.16 GB
  ETA 14k×3:     1.0h best / 1.2h worst

  Sample outputs (first 3):
    [0] {"spans": []}<|im_end|>
    [1] {"spans": []}<|im_end|>
    [2] {"spans": ["History of hypertension, currently on lisinopril."]}<|im_end|>
  VRAM free after engine destroy: 2.0 GB
  Deleted cache: models--meta-llama--Llama-3.1-8B-Instruct  (29.9 GB freed)
  Disk free after cache delete:   19.5 GB
(Worker pid=5031) (Worker_TP0 pid=5031) INFO 05-01 16:34:19 [multiproc_executor.py:749] Parent process exited, terminating worker
(Worker pid=5032) (Worker_TP1 pid=5032) INFO 05-01 16:34:19 [multiproc_executor.py:749] Parent process exited, terminating worker
(Worker pid=5031) (Worker_TP0 pid=5031) INFO 05-01 16:34:19 [multiproc_executor.py:802] WorkerProc shutting down.
(Worker pid=5032) (Worker_TP1 pid=5032) INFO 05-01 16:34:19 [multiproc_executor.py:802] WorkerProc shutting down.
  ✓ Saved results → /kaggle/workin

## Section 4 — Results Summary

Ranked by output tok/s (higher = better).
9-hour budget line shown — 3 models run sequentially, each needs to process 14,300 rows.

In [8]:
with open(RESULTS_PATH) as f:
    all_results = json.load(f)

rows = []
for r in all_results:
    status = r["status"]
    rows.append({
        "Model":            r["label"],
        "Status":           status,
        "VRAM load (GB)":   r.get("vram_model_gb"),
        "VRAM peak (GB)":   r.get("vram_peak_gb"),
        "Output tok/s":     r.get("output_toks_ps"),
        "ETA 14k (h)": r.get("eta_14k_hours"),
        "ETA worst (h)":    r.get("eta_worst_hours"),
        "Fits 9h budget?":  (
            "✓" if r.get("eta_worst_hours") is not None and r["eta_worst_hours"] < 9
            else ("✗" if r.get("eta_worst_hours") is not None else "ERR")
        ),
        "Error":            (r.get("error") or "")[:80],
    })

df = pd.DataFrame(rows)
ok = df[df["Status"] == "OK"].sort_values("Output tok/s", ascending=False)
err = df[df["Status"] != "OK"]

print("\n" + "="*80)
print("  WORKING MODELS (ranked by throughput)")
print("="*80)
if not ok.empty:
    print(ok[["Model", "VRAM load (GB)", "VRAM peak (GB)", "Output tok/s",
               "ETA 14k (h)", "ETA worst (h)", "Fits 9h budget?"]].to_string(index=False))
else:
    print("  No models completed successfully.")

if not err.empty:
    print("\n" + "="*80)
    print("  FAILED MODELS")
    print("="*80)
    print(err[["Model", "Status", "Error"]].to_string(index=False))

print("\n9-hour budget line: each of 3 models must finish 14,300 rows in <3h individually")
print("(or combined total <9h if models can share resources)")


  WORKING MODELS (ranked by throughput)
                                        Model  VRAM load (GB)  VRAM peak (GB)  Output tok/s  ETA 14k (h)  ETA worst (h) Fits 9h budget?
meta-llama/Llama-3.2-1B-Instruct (fp16, TP=1)           13.26           13.47        1376.5         0.35           0.43               ✓
                      Qwen3-0.6B (fp16, TP=1)           13.33           13.34        1155.7         0.41           0.52               ✓
                      Qwen3-1.7B (fp16, TP=1)           13.31           13.52         998.8         0.48           0.60               ✓
                        Qwen3-8B (fp16, TP=2)           27.00           27.21         561.1         0.85           1.06               ✓
   LiquidAI/LFM2.5-1.2B-Instruct (fp16, TP=1)           13.05           13.71         505.9         0.94           1.18               ✓
                    Llama-3.1-8B (fp16, TP=2)           26.94           27.16         500.3         0.95           1.19               ✓
       

## Section 5 — Output Quality Check

Verify models produce valid JSON `{"spans": [...]}` format.
Measures JSON parse success rate on the bench outputs.

In [9]:
import re

QUALITY_RESULTS_PATH = Path("/kaggle/working/quality_results.json")

def check_json_validity(text: str) -> bool:
    """Try to parse JSON output, with think-block stripping and regex fallback."""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(text)
        return isinstance(parsed.get("spans"), list)
    except Exception:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group())
                return isinstance(parsed.get("spans"), list)
            except Exception:
                pass
    return False

QUALITY_N = 20
quality_prompts = ALL_PROMPTS[:QUALITY_N]

# Load existing quality results if resuming
if QUALITY_RESULTS_PATH.exists():
    with open(QUALITY_RESULTS_PATH) as f:
        quality_results = json.load(f)
    print(f"Resuming quality — already done: {list(quality_results.keys())}")
else:
    quality_results = {}

for candidate in CANDIDATES:
    label    = candidate["label"]
    model_id = candidate["model_id"]
    tp       = candidate["tp"]
    dtype    = candidate["dtype"]

    # Skip if already done
    if label in quality_results:
        print(f"SKIP quality (already done): {label}")
        continue

    # Skip if model failed in benchmark
    bench_res = next((r for r in all_results if r["label"] == label), {})
    if bench_res.get("status") not in ("OK", None, "PENDING"):
        print(f"SKIP quality (failed in benchmark): {label}")
        quality_results[label] = {"valid_pct": None, "error": bench_res.get("error")}
        with open(QUALITY_RESULTS_PATH, "w") as f:
            json.dump(quality_results, f, indent=2)
        continue

    print(f"\n{'='*50}")
    print(f"Quality check: {label}")
    print(f"  Disk free: {disk_free_gb():.1f} GB  |  VRAM used: {get_vram_used_gb():.2f} GB")
    print(f"{'='*50}")

    attn_cfg = AttentionConfig(backend=AttentionBackendEnum.TRITON_ATTN)
    llm = None
    try:
        llm = LLM(
            model                  = model_id,
            dtype                  = dtype,
            tensor_parallel_size   = tp,
            gpu_memory_utilization = GPU_MEM_UTIL,
            max_model_len          = MAX_IN_TOKS + MAX_OUT_TOKS,
            enforce_eager          = True,
            trust_remote_code      = False,
            seed                   = 42,
            download_dir           = str(MODEL_CACHE_DIR),
            attention_config       = attn_cfg,
        )
        sp = SamplingParams(temperature=0.0, max_tokens=MAX_OUT_TOKS)
        outputs = llm.generate(quality_prompts, sp, use_tqdm=False)

        texts       = [req.outputs[0].text.strip() for req in outputs]
        valid_count = sum(check_json_validity(t) for t in texts)
        valid_pct   = round(valid_count / len(texts) * 100, 1)
        quality_results[label] = {"valid_pct": valid_pct, "samples": texts[:3]}

        print(f"  JSON valid: {valid_count}/{len(texts)} ({valid_pct}%)")
        for i, t in enumerate(texts[:3]):
            print(f"  [{i}] {t[:120]}")

    except Exception as e:
        quality_results[label] = {"valid_pct": None, "error": str(e)[:200]}
        print(f"  FAILED: {e}")

    finally:
        # Always clean up GPU + disk regardless of success/failure
        if llm is not None:
            destroy_engine(llm)
        print(f"  VRAM free after destroy: {get_vram_free_gb():.1f} GB")
        delete_model_cache(model_id)
        print(f"  Disk free after cache delete: {disk_free_gb():.1f} GB")

    # Save after each model (resume-safe)
    with open(QUALITY_RESULTS_PATH, "w") as f:
        json.dump(quality_results, f, indent=2)
    print(f"  ✓ Saved quality results → {QUALITY_RESULTS_PATH}")

print("\n✓ Quality check complete")


Quality check: Qwen3-0.6B (fp16, TP=1)
  Disk free: 19.5 GB  |  VRAM used: 0.21 GB
INFO 05-01 16:34:24 [utils.py:238] non-default args: {'download_dir': '/kaggle/working/model_cache', 'dtype': 'half', 'seed': 42, 'max_model_len': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=True, use_trtllm_attention=None, disable_flashinfer_prefill=False, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False)}
INFO 05-01 16:34:24 [model.py:531] Resolved architecture: Qwen3ForCausalLM
WARNING 05-01 16:34:24 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 05-01 16:34:24 [model.py:1554] Using max model len 2048
INFO 05-01 16:34:24 [scheduler.py:231] 

2026-05-01 16:34:36.844713: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777653276.871140    5742 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777653276.879156    5742 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777653276.898200    5742 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777653276.898234    5742 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777653276.898237    5742 computation_placer.cc:177] computation placer alr

(EngineCore_DP0 pid=5742) INFO 05-01 16:34:44 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir='/kaggle/working/model_cache', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_c

[W501 16:34:46.306843643 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore_DP0 pid=5742) INFO 05-01 16:34:47 [base.py:106] Offloader set to NoopOffloader
(EngineCore_DP0 pid=5742) INFO 05-01 16:34:47 [gpu_model_runner.py:4281] Starting to load model Qwen/Qwen3-0.6B...
(EngineCore_DP0 pid=5742) INFO 05-01 16:34:47 [cuda.py:368] Using AttentionBackendEnum.TRITON_ATTN backend.
(EngineCore_DP0 pid=5742) INFO 05-01 16:34:55 [weight_utils.py:561] Time spent downloading weights for Qwen/Qwen3-0.6B: 6.900993 seconds
(EngineCore_DP0 pid=5742) INFO 05-01 16:34:55 [weight_utils.py:601] No model.safetensors.index.json found in remote.


...

(Worker pid=10178) INFO 05-01 16:44:00 [parallel_state.py:1393] world_size=2 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:59649 backend=nccl
(Worker pid=10179) INFO 05-01 16:44:01 [parallel_state.py:1393] world_size=2 rank=1 local_rank=1 distributed_init_method=tcp://127.0.0.1:59649 backend=nccl


(Worker pid=10179) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=10178) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=10178) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=10179) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(Worker pid=10178) INFO 05-01 16:44:01 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=10178) WARNING 05-01 16:44:02 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=10179) WARNING 05-01 16:44:02 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=10179) INFO 05-01 16:44:02 [parallel_state.py:1715] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP rank N/A, EPLB rank N/A
(Worker pid=10178) INFO 05-01 16:44:02 [parallel_state.py:1715] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(Worker pid=10178) INFO 05-01 16:44:02 [base.py:106] Offloader set to NoopOffloader
(Worker pid=10178) (Worker_TP0 pid=10178) INFO 05-01 16:44:02 [gpu_model_runner.py:4281] Starting to load model meta-llama/Llama-3.1-8B-Instruct...
(Worker pid=10179) INFO 05-01 16:44:02 [ba

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:11<00:35, 11.90s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:22<00:22, 11.39s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:37<00:12, 12.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:42<00:00,  9.57s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:42<00:00, 10.51s/it]
(Worker pid=10178) (Worker_TP0 pid=10178) 


(Worker pid=10178) (Worker_TP0 pid=10178) INFO 05-01 16:45:58 [default_loader.py:293] Loading weights took 42.09 seconds
(Worker pid=10178) (Worker_TP0 pid=10178) INFO 05-01 16:45:59 [gpu_model_runner.py:4364] Model loading took 7.51 GiB memory and 115.021340 seconds
(Worker pid=10178) (Worker_TP0 pid=10178) INFO 05-01 16:46:09 [gpu_worker.py:424] Available KV cache memory: 5.04 GiB
(EngineCore_DP0 pid=10153) INFO 05-01 16:46:09 [kv_cache_utils.py:1314] GPU KV cache size: 82,592 tokens
(EngineCore_DP0 pid=10153) INFO 05-01 16:46:09 [kv_cache_utils.py:1319] Maximum concurrency for 2,048 tokens per request: 40.33x
(EngineCore_DP0 pid=10153) INFO 05-01 16:46:09 [core.py:282] init engine (profile, create kv cache, warmup model) took 9.98 seconds
(EngineCore_DP0 pid=10153) INFO 05-01 16:46:13 [vllm.py:747] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=10153) WARNING 05-01 16:46:13 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to settin

## Section 6 — Final Decision Table

Combined throughput + quality ranking.
Pick 3 models for ensemble: diversity (different checkpoints) + all fit 9h budget.

In [10]:
with open(RESULTS_PATH) as f:
    all_results = json.load(f)
with open(QUALITY_RESULTS_PATH) as f:
    quality_results = json.load(f)

rows = []
for r in all_results:
    if r["status"] != "OK":
        continue
    q = quality_results.get(r["label"], {})
    rows.append({
        "Model":           r["label"],
        "Output tok/s":    r.get("output_toks_ps"),
        "ETA worst (h)":   r.get("eta_worst_hours"),
        "JSON valid %":    q.get("valid_pct"),
        "VRAM model (GB)": r.get("vram_model_gb"),
        "VRAM peak (GB)":  r.get("vram_peak_gb"),
        "Fits 9h?":        "✓" if (r.get("eta_worst_hours") or 99) < 9 else "✗",
    })

df_final = pd.DataFrame(rows).sort_values("Output tok/s", ascending=False)

print("\n" + "="*90)
print("  FINAL RANKING — Throughput × Quality")
print("="*90)
print(df_final.to_string(index=False))

print("\n" + "─"*90)
print("Ensemble recommendation: pick top 3 rows where:")
print("  1. Fits 9h budget ✓")
print("  2. JSON valid % >= 90%")
print("  3. Models are architecturally diverse (not all same checkpoint)")
print("─"*90)


  FINAL RANKING — Throughput × Quality
                                        Model  Output tok/s  ETA worst (h)  JSON valid %  VRAM model (GB)  VRAM peak (GB) Fits 9h?
meta-llama/Llama-3.2-1B-Instruct (fp16, TP=1)        1376.5           0.43           0.0            13.26           13.47        ✓
                      Qwen3-0.6B (fp16, TP=1)        1155.7           0.52         100.0            13.33           13.34        ✓
                      Qwen3-1.7B (fp16, TP=1)         998.8           0.60         100.0            13.31           13.52        ✓
                        Qwen3-8B (fp16, TP=2)         561.1           1.06          95.0            27.00           27.21        ✓
   LiquidAI/LFM2.5-1.2B-Instruct (fp16, TP=1)         505.9           1.18         100.0            13.05           13.71        ✓
                    Llama-3.1-8B (fp16, TP=2)         500.3           1.19         100.0            26.94           27.16        ✓
                        Qwen3-4B (fp16, TP=